## **09 -- Temporal validation with an approximate calendar-year split**

**Place in the framework.** This notebook carries out gate-P4 criterion C4 (stability over time) inside stage E9, for the main model
and for the models of returns R4 and R2. Each section below starts with a **Framework** line.

| Section | Framework | What it produces |
|---|---|---|
| 1 | E1 | Fixed configuration and reference counts |
| 2 | E7 · E4 / P1 | Predictor dictionary and roles; check against the frozen reference counts; escalation label of return R2 |
| 3 | E9 · C4 · E5 | Temporal partitions; no patient in both periods |
| 4-6 | E8 | Nested tuning, stacking, recalibration, metrics, bootstrap, reports |
| 7 | E9 · C4 | Development on the earlier period, single test on the later period: `cut2017`, `cut2015`, `cut2017_landmark`, `cut2017_landmark_labs` (R4), `cut2017_escalation` (R2) |
| 8 | E11 · P4 | Summary table for the manuscript and for the gate-P4 classification |

Framework stages: E1 Governance and configuration · E2 Universe and population · E3 Outcome definition · E4 Count audit ·
E5 Unit of analysis and overlap · E6 Prediction point and window · E7 Predictors · E8 Modeling, validation and metrics ·
E9 Sensitivity and robustness · E10 From classification to action · E11 Reporting. Gates P1-P4; returns R1-R4; gate-P4 criteria C1-C4
(C1 events already present, C2 outcome with time, C3 detection bias, C4 stability over time).

**Purpose.** MIMIC-IV shifts every patient's dates and only reports a 3-year `anchor_year_group` per patient, so an admission's real year
is known only as an interval. `mimiciv.timing_flags_v3` (`sql/07_timing_audit/01_timing_flags_v3.sql`) gives each admission an
approximate real-year interval `[year_lo, year_hi]` = `[group start + D, group end + D]`, with `D = year(admittime) - anchor_year`.

**Design.** For a cut year `C`:

* **development set** = admissions with `year_hi < C` (certainly before `C`);
* **test set** = admissions with `year_lo >= C` (certainly from `C` on);
* admissions whose interval straddles `C` are **discarded**;
* test admissions of patients who also have an admission in the development set are **removed** (no patient on both sides).

The models are developed **only** on the development set with the same pipeline as notebooks 02 and 08 (patient-grouped inner tuning,
out-of-fold stacking, Platt recalibration on development patients) and then applied **once** to the later test set. The development is
repeated `N_REPEATS` times with different seeds (inner folds, forests); metrics are averaged over repeats. Intervals come from a
patient-level bootstrap of the test set (`BOOT_SENS` replicates).

| Analysis | Cut | Population |
|---|---|---|
| `cut2017` (main) | 2017 | eligible delivery hospitalizations |
| `cut2015` (sensitivity) | 2015 | eligible delivery hospitalizations (cut coincides with the ICD-9 to ICD-10 transition) |
| `cut2017_landmark` | 2017 | landmark population of notebook 08 (no ICU before T; SMM of timing classes 1 and 2 removed) |
| `cut2017_landmark_labs` | 2017 | landmark population with the extended laboratory set (return R4) |
| `cut2017_escalation` | 2017 | eligible deliveries without ICU before T; outcome = escalation of care after T (return R2) |

The rise of the SMM rate between periods is expected to show up as calibration drift (observed/expected above 1 in the test set).

**Data-use rules.** Only aggregate results are printed; files written: `data/outputs_<analysis>.txt` and `data/outputs_temporal_summary.txt`.

**Run order.** Run `sql/07_timing_audit/01_timing_flags_v3.sql` first, then this notebook top to bottom: `QUICK = True` for a smoke test
(never report QUICK numbers), then `QUICK = False`.

### **1. Libraries, connection and settings**

> **Framework:** E1 Governance and configuration. Fixed seed, repeats and bootstrap replicates (D1); reference counts frozen at gate P1.

In [11]:
import os, sys, time, math, warnings
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# notebooks/ -> project root is one level up
sys.path.append(os.path.abspath(os.path.join("..")))
import libraries.database as db_lib
engine = db_lib.get_connect_database()

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

# ----------------------------------------------------------------------------- settings
QUICK        = False         # True = smoke test (small forests, 1 repeat, B = 50). Never report QUICK numbers.
SEED         = 20260101
OUTER_FOLDS  = 5
N_REPEATS    = 3            # repeats of the outer grouped CV (development instability)
INNER_FOLDS  = 3
BOOT         = 1000         # cluster bootstrap replicates for the main analysis
BOOT_SENS    = 300          # replicates for the sensitivity analyses
N_JOBS       = -1
N_TREES      = 300
XGB_TREES    = 200
APPLY_LIMITS = True         # set values outside the plausibility limits of the dictionary to missing
CAT_COLS     = ["ethnicity", "insurance", "language", "admission_type", "admission_location"]
DCA_T        = [0.005, 0.01, 0.02, 0.03, 0.05, 0.10, 0.15, 0.20]
EXPECTED     = {"eligible_deliveries": 19383, "smm": 445, "deaths": 3}   # from the audit counts (Q-F / Q-G)

if QUICK:
    OUTER_FOLDS, N_REPEATS, BOOT, BOOT_SENS = 3, 1, 50, 30
    N_TREES, XGB_TREES = 60, 50
print(f"QUICK={QUICK}  outer={OUTER_FOLDS}x{N_REPEATS}  inner={INNER_FOLDS}  B={BOOT}  trees={N_TREES}")

QUICK=False  outer=5x3  inner=3  B=1000  trees=300


### **2. Load the predictor table and the final dictionary**

> **Framework:** E7 Predictors and E4 / P1 (check against the reference counts). Section 2 also loads the escalation label of return R2.

`predictors_24h_v2` has one row per obstetric hospitalization with the predictors observed in the window
(admission - 24 h, admission + 24 h). `predictor_dictionary_final_v2` assigns every variable its role
(`main model`, `ICU sensitivity only`, `laboratory sensitivity ...`) and its plausibility limits.
Only counts are printed.

In [12]:
df = pd.read_sql("SELECT * FROM mimiciv.predictors_24h_v2", engine)
dic = pd.read_sql("""SELECT variable, model_role_final, missingness_indicator_main, plausible_low, plausible_high,
                            n_eligible_delivery, coverage_pct
                     FROM mimiciv.predictor_dictionary_final_v2""", engine)

for c in ["eligible_24h", "icu_in_24h", "is_delivery_v2", "smm_any", "died_same_admission"]:
    df[c] = df[c].fillna(False).astype(bool)
df["admittime"] = pd.to_datetime(df["admittime"])

VAR_MAIN = dic.loc[dic.model_role_final == "main model", "variable"].tolist()
VAR_ICU  = dic.loc[dic.model_role_final == "ICU sensitivity only", "variable"].tolist()
VAR_LAB  = dic.loc[dic.model_role_final.str.startswith("laboratory sensitivity"), "variable"].tolist()
IND_MAIN = dic.loc[dic.missingness_indicator_main.fillna(False).astype(bool), "variable"].tolist()

print("Dictionary roles ->", {"main": VAR_MAIN, "ICU only": len(VAR_ICU), "lab sensitivity": len(VAR_LAB),
                              "missingness indicator in main model": IND_MAIN})

# Plausibility limits (proposals pending clinical confirmation): out-of-range values become missing
if APPLY_LIMITS:
    n_set = {}
    for _, r in dic.dropna(subset=["plausible_low", "plausible_high"]).iterrows():
        v = r["variable"]
        if v in df.columns:
            bad = (df[v] < r["plausible_low"]) | (df[v] > r["plausible_high"])
            n_set[v] = int(bad.sum())
            df.loc[bad, v] = np.nan
    print("Values set to missing by plausibility limits (count per variable):", n_set)

# Missingness indicators (computed row by row from the raw value: no information crosses patients)
for v in set(IND_MAIN + VAR_LAB + VAR_ICU):
    df[f"{v}_missing"] = df[v].isna().astype(int)
df["icu_in_24h_bin"] = df["icu_in_24h"].astype(int)
df["y"] = df["smm_any"].astype(int)

# Audit checks against the counts already certified (aggregate only)
elig = df[df.eligible_24h & df.is_delivery_v2]
chk = {"eligible_deliveries": len(elig), "smm": int(elig.smm_any.sum()), "deaths": int(elig.died_same_admission.sum())}
for k, v in chk.items():
    print(f"{k}: {v}  (expected {EXPECTED[k]})  [{'OK' if v == EXPECTED[k] else 'CHECK: differs from the audit counts'}]")
print("Eligible any obstetric admissions:", int(df.eligible_24h.sum()),
      "| SMM:", int(df[df.eligible_24h].smm_any.sum()), "| deaths:", int(df[df.eligible_24h].died_same_admission.sum()))

Dictionary roles -> {'main': ['hemoglobin', 'creatinine'], 'ICU only': 14, 'lab sensitivity': 11, 'missingness indicator in main model': ['creatinine']}
Values set to missing by plausibility limits (count per variable): {'hemoglobin': 0, 'hematocrit': 0, 'creatinine': 1}
eligible_deliveries: 19383  (expected 19383)  [OK]
smm: 445  (expected 445)  [OK]
deaths: 3  (expected 3)  [OK]
Eligible any obstetric admissions: 23625 | SMM: 814 | deaths: 5


In [13]:
# Approximate real-year interval and timing class of every eligible delivery hospitalization
tf = pd.read_sql("""SELECT hadm_id, year_lo, year_hi, smm_timing_class
                     FROM mimiciv.timing_flags_v3""", engine)
df = df.drop(columns=[x for x in tf.columns if x != "hadm_id" and x in df.columns]).merge(tf, on="hadm_id", how="left")
df["smm_timing_class"] = df["smm_timing_class"].fillna("not an eligible delivery")
e = df[df.eligible_24h & df.is_delivery_v2]
print(f"Eligible deliveries: {len(e)} (expected {EXPECTED['eligible_deliveries']}); "
      f"with a year interval: {int(e.year_lo.notna().sum())}")

# Escalation of care after T (timed outcome of return R2; sql/07_timing_audit/04_escalation_label.sql)
EXPECTED_ESC = {"population": 19268, "escalation": 101}
esc = pd.read_sql("SELECT hadm_id, escalation_any FROM mimiciv.escalation_v3", engine)
esc["escalation_any"] = esc["escalation_any"].fillna(False).astype(bool)
print(f"escalation population: {len(esc)} (expected {EXPECTED_ESC['population']}); "
      f"events: {int(esc.escalation_any.sum())} (expected {EXPECTED_ESC['escalation']})  "
      f"[{'OK' if (len(esc), int(esc.escalation_any.sum())) == tuple(EXPECTED_ESC.values()) else 'CHECK'}]")
df = df.drop(columns=[x for x in ("escalation_any", "in_esc_pop", "y_esc") if x in df.columns]).merge(esc, on="hadm_id", how="left")
df["in_esc_pop"] = df["escalation_any"].notna()
df["y_esc"] = df["escalation_any"].fillna(False).astype(int)

Eligible deliveries: 19383 (expected 19383); with a year interval: 19383
escalation population: 19268 (expected 19268); events: 101 (expected 101)  [OK]


### **3. Temporal partitions**

> **Framework:** E9 · gate-P4 criterion C4. Development and test periods from the approximate real-year interval; test patients present in development are removed (grouping unit of E5).

`make_temporal(name)` returns the development and test sets of one analysis. The table printed after it gives the counts behind each
partition (expected for `cut2017` from query Q-T4: 11,104 admissions / 209 SMM before, 6,315 / 201 from 2017 on, 1,964 straddling;
the test counts printed here are lower because test admissions of patients already in the development set are removed).

The last two analyses feed the second gate-P4 classification (criterion C4 for the models of the returns):
`cut2017_landmark_labs` uses the landmark population with the extended laboratory set (return R4), and `cut2017_escalation` uses
escalation of care after T as the outcome (return R2; label from `sql/07_timing_audit/04_escalation_label.sql`, run it first).
`TEMPORAL_RUN` in section 7 lists the analyses to run; remove names from it to run only some of them.

In [14]:
TEMPORAL = {
    "cut2017":          dict(cut=2017, landmark=False),
    "cut2015":          dict(cut=2015, landmark=False),
    "cut2017_landmark": dict(cut=2017, landmark=True),
    # second gate-P4 classification (criterion C4 for the models of the returns)
    "cut2017_landmark_labs": dict(cut=2017, landmark=True, labs=True),   # return R4: extended laboratory set
    "cut2017_escalation":    dict(cut=2017, landmark=False, escalation=True),  # return R2: escalation after T
}
EXPECTED_Q_T4 = {2017: (11104, 209, 6315, 201, 1964), 2015: (8017, 121, 8279, 236, 3087)}

def make_temporal(name):
    cfg = TEMPORAL[name]; C = cfg["cut"]
    d = df[df.eligible_24h & df.is_delivery_v2].copy()
    label = "y"
    if cfg.get("escalation"):
        d = d[d.in_esc_pop & ~d.icu_in_24h]
        label = "y_esc"
    if cfg["landmark"]:
        d = d[~d.icu_in_24h & ~d.smm_timing_class.isin(["1 prevalent", "2 indeterminate"])]
    before, after = d.year_hi < C, d.year_lo >= C
    dev, test_all = d[before], d[after]
    shared = test_all.subject_id.isin(set(dev.subject_id))
    test = test_all[~shared]
    num  = [c for c in dict.fromkeys(["age_at_admission"] + VAR_MAIN) if c in d.columns]
    binc = [c for c in dict.fromkeys(f"{v}_missing" for v in IND_MAIN) if c in d.columns and dev[c].nunique() > 1]
    if cfg.get("labs"):
        num  = list(dict.fromkeys(num + [v for v in VAR_LAB if v in d.columns]))
        binc = list(dict.fromkeys(binc + [f"{v}_missing" for v in VAR_LAB
                                          if f"{v}_missing" in d.columns and dev[f"{v}_missing"].nunique() > 1]))
    cat  = [c for c in CAT_COLS if c in d.columns]
    info = dict(cut=C, population=len(d), dev_adm=len(dev), dev_pat=dev.subject_id.nunique(), dev_events=int(dev[label].sum()),
                straddling=int((~before & ~after).sum()), straddling_events=int(d[~before & ~after][label].sum()),
                test_adm_all=len(test_all), test_events_all=int(test_all[label].sum()),
                test_removed_shared=int(shared.sum()), test_removed_shared_events=int(test_all[shared][label].sum()),
                test_adm=len(test), test_pat=test.subject_id.nunique(), test_events=int(test[label].sum()))
    return dict(name=name, train=dev.reset_index(drop=True), test=test.reset_index(drop=True),
                num=num, bin=binc, cat=cat, label=label, info=info)

rows = []
for nm in TEMPORAL:
    i = make_temporal(nm)["info"]
    rows.append({"analysis": nm, "cut": i["cut"],
                 "dev adm (SMM)": f"{i['dev_adm']} ({i['dev_events']})",
                 "dev prev %": round(100 * i["dev_events"] / max(i["dev_adm"], 1), 2),
                 "straddling (SMM)": f"{i['straddling']} ({i['straddling_events']})",
                 "test before patient removal (SMM)": f"{i['test_adm_all']} ({i['test_events_all']})",
                 "removed: patient in dev (SMM)": f"{i['test_removed_shared']} ({i['test_removed_shared_events']})",
                 "test adm (SMM)": f"{i['test_adm']} ({i['test_events']})",
                 "test prev %": round(100 * i["test_events"] / max(i["test_adm"], 1), 2)})
    if not TEMPORAL[nm]["landmark"] and not TEMPORAL[nm].get("escalation"):
        exp = EXPECTED_Q_T4[i["cut"]]
        got = (i["dev_adm"], i["dev_events"], i["test_adm_all"], i["test_events_all"], i["straddling"])
        print(f"{nm}: counts vs Q-T4 {got} expected {exp}  [{'OK' if got == exp else 'CHECK'}]")
print(pd.DataFrame(rows).to_string(index=False))

cut2017: counts vs Q-T4 (11104, 209, 6315, 201, 1964) expected (11104, 209, 6315, 201, 1964)  [OK]
cut2015: counts vs Q-T4 (8017, 121, 8279, 236, 3087) expected (8017, 121, 8279, 236, 3087)  [OK]
             analysis  cut dev adm (SMM)  dev prev % straddling (SMM) test before patient removal (SMM) removed: patient in dev (SMM) test adm (SMM)  test prev %
              cut2017 2017   11104 (209)        1.88        1964 (35)                        6315 (201)                     1240 (28)     5075 (173)         3.41
              cut2015 2015    8017 (121)        1.51        3087 (88)                        8279 (236)                     1104 (22)     7175 (214)         2.98
     cut2017_landmark 2017   11008 (127)        1.15        1948 (21)                        6242 (137)                     1225 (20)     5017 (117)         2.33
cut2017_landmark_labs 2017   11008 (127)        1.15        1948 (21)                        6242 (137)                     1225 (20)     5017 (117)        

### **4. Core functions (preprocessing, learners, nested tuning, stacking, calibration)**

> **Framework:** E8 Modeling, validation and metrics (same functions as notebooks 02 and 08).

Everything that learns from data is inside the training portion of each split:

* `Winsor` clips each numeric column to the 0.5th-99.5th percentiles *of the training data*.
* The median imputer, the scaler and the one-hot encoder are fitted inside a `Pipeline`, once per fit.
* Hyperparameters are chosen by **inner patient-grouped CV** (average precision of the pooled out-of-fold predictions).
* The meta-learner of the ensemble is trained on the **out-of-fold** predictions of the base learners (never on in-sample predictions).
* The Platt calibrator is trained on out-of-fold predictions of the training patients; the outer test patients are only scored.

In [15]:
def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def expit(z):
    return 1.0 / (1.0 + np.exp(-z))

class Winsor(BaseEstimator, TransformerMixin):
    """Clip each column to training percentiles (fit on the training data only)."""
    def __init__(self, lo=0.5, hi=99.5):
        self.lo = lo; self.hi = hi
    def fit(self, X, y=None):
        X = np.asarray(X, float)
        self.lo_ = np.nanpercentile(X, self.lo, axis=0)
        self.hi_ = np.nanpercentile(X, self.hi, axis=0)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, float), self.lo_, self.hi_)

def make_prep(num, binc, cat):
    parts = [("num", Pipeline([("w", Winsor()),
                               ("imp", SimpleImputer(strategy="median", keep_empty_features=True)),
                               ("sc", StandardScaler())]), num)]
    if binc:
        parts.append(("bin", "passthrough", binc))
    if cat:
        parts.append(("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                                       ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                            min_frequency=0.01, sparse_output=False))]), cat))
    return ColumnTransformer(parts)

def prep_frame(d, num, binc, cat):
    X = d[num + binc + cat].copy()
    for c in cat:
        X[c] = X[c].astype(object).where(X[c].notna(), np.nan)
    return X

# Small prespecified grids (class weights are options; no resampling)
GRID = {
    # The grids cover values beyond those most often chosen in preliminary runs, so that the selected value
    # is not at the edge of the grid (ET depth, RF depth and XGB depth; class weights remain options).
    "LR":  [dict(C=C, cw=cw) for C in (0.01, 0.1, 1.0) for cw in (None, "balanced")],
    "RF":  [dict(depth=dp, cw=cw) for dp in (2, 4, 8) for cw in (None, "balanced_subsample")],
    "ET":  [dict(depth=dp, cw=cw) for dp in (4, 8, 12) for cw in (None, "balanced_subsample")],
    "XGB": [dict(depth=dp, spw=sp) for dp in (1, 2, 4) for sp in (1, "sqrt")],
}
MODELS = ["LR", "RF", "ET", "XGB", "Stack", "Stack_raw"]
BOOT_MODELS = ["LR", "RF", "ET", "XGB", "Stack"]
PAIRS = [("RF", "LR"), ("ET", "LR"), ("XGB", "LR"), ("Stack", "LR"), ("Stack", "XGB")]

def cfg_str(c):
    return ",".join(f"{k}={v}" for k, v in c.items())

def make_clf(name, cfg, ytr, seed):
    if name == "LR":
        return LogisticRegression(C=cfg["C"], class_weight=cfg["cw"], max_iter=3000)
    if name == "RF":
        return RandomForestClassifier(n_estimators=N_TREES, max_depth=cfg["depth"], min_samples_leaf=20,
                                      class_weight=cfg["cw"], n_jobs=N_JOBS, random_state=seed)
    if name == "ET":
        return ExtraTreesClassifier(n_estimators=N_TREES, max_depth=cfg["depth"], min_samples_leaf=20,
                                    class_weight=cfg["cw"], n_jobs=N_JOBS, random_state=seed)
    if name == "XGB":
        pos = max(int(ytr.sum()), 1); neg = max(len(ytr) - pos, 1)
        spw = math.sqrt(neg / pos) if cfg["spw"] == "sqrt" else 1.0
        return XGBClassifier(n_estimators=XGB_TREES, learning_rate=0.05, max_depth=cfg["depth"], subsample=0.8,
                             colsample_bytree=0.8, min_child_weight=5, scale_pos_weight=spw, eval_metric="logloss",
                             tree_method="hist", n_jobs=N_JOBS if N_JOBS and N_JOBS > 0 else 0, random_state=seed)
    raise ValueError(name)

def fit_predict(name, cfg, prep, Xtr, ytr, Xte, seed):
    pipe = Pipeline([("prep", clone(prep)), ("clf", make_clf(name, cfg, ytr, seed))])
    pipe.fit(Xtr, ytr)
    return pipe.predict_proba(Xte)[:, 1]

def platt(o, t, ytr):
    c = LogisticRegression(penalty=None, max_iter=1000).fit(logit(o).reshape(-1, 1), ytr)
    return c.predict_proba(logit(t).reshape(-1, 1))[:, 1]

def tune_stack_calibrate(Xtr, ytr, gtr, Xte, prep, seed):
    """Inside one outer training set: tune, build out-of-fold predictions, stack, calibrate; score the outer test set."""
    inner = list(StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=seed).split(Xtr, ytr, gtr))
    names = list(GRID)
    chosen, oof, pred_te = {}, {}, {}
    for name in names:
        best = None
        for cfg in GRID[name]:
            o = np.zeros(len(ytr))
            for tr, va in inner:
                o[va] = fit_predict(name, cfg, prep, Xtr.iloc[tr], ytr[tr], Xtr.iloc[va], seed)
            ap = average_precision_score(ytr, o)
            if best is None or ap > best[0]:
                best = (ap, cfg, o)
        chosen[name] = best[1]; oof[name] = best[2]
        pred_te[name] = fit_predict(name, best[1], prep, Xtr, ytr, Xte, seed)
    # explicit stacking: logistic meta-learner on out-of-fold base predictions (C = 0.1 as in the original framework)
    Z_oof = np.column_stack([logit(oof[n]) for n in names])
    Z_te = np.column_stack([logit(pred_te[n]) for n in names])
    meta = LogisticRegression(C=0.1, max_iter=2000).fit(Z_oof, ytr)
    stack_te_raw = meta.predict_proba(Z_te)[:, 1]
    # cross-fitted meta out-of-fold predictions -> calibrator never sees the predictions of its own training rows
    mo = np.zeros(len(ytr))
    for tr, va in StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=seed + 1).split(Z_oof, ytr, gtr):
        m = LogisticRegression(C=0.1, max_iter=2000).fit(Z_oof[tr], ytr[tr])
        mo[va] = m.predict_proba(Z_oof[va])[:, 1]
    out = {n: platt(oof[n], pred_te[n], ytr) for n in names}
    out["Stack"] = platt(mo, stack_te_raw, ytr)
    out["Stack_raw"] = stack_te_raw
    return out, chosen

def run_cv(spec, label=""):
    d = spec["data"]
    X = prep_frame(d, spec["num"], spec["bin"], spec["cat"])
    y = d[spec["label"]].values.astype(int)
    g = d["subject_id"].values
    prep = make_prep(spec["num"], spec["bin"], spec["cat"])
    n = len(y)
    preds = {m: np.full((N_REPEATS, n), np.nan) for m in MODELS}
    chosen_log = []
    split_log = []
    t0 = time.time(); done = 0; total = N_REPEATS * OUTER_FOLDS
    for r in range(N_REPEATS):
        outer = StratifiedGroupKFold(OUTER_FOLDS, shuffle=True, random_state=SEED + r)
        for k, (tr, te) in enumerate(outer.split(X, y, g)):
            assert len(set(g[tr]) & set(g[te])) == 0, "patient present in training and test"
            split_log.append(dict(repeat=r + 1, fold=k + 1,
                                  train_adm=len(tr), train_pat=len(set(g[tr])), train_events=int(y[tr].sum()),
                                  train_event_pat=len(set(g[tr][y[tr] == 1])),
                                  test_adm=len(te), test_pat=len(set(g[te])), test_events=int(y[te].sum()),
                                  test_event_pat=len(set(g[te][y[te] == 1])), shared_patients=0))
            out, chosen = tune_stack_calibrate(X.iloc[tr], y[tr], g[tr], X.iloc[te], prep, SEED + 100 * r + k)
            for m in MODELS:
                preds[m][r, te] = out[m]
            for nme, cfg in chosen.items():
                chosen_log.append((nme, cfg_str(cfg)))
            done += 1
            if done == 1:
                el = time.time() - t0
                print(f"[{spec['name']}] first outer fold took {el/60:.1f} min -> projected total about {el*total/60:.0f} min")
    print(f"[{spec['name']}] cross-validation finished in {(time.time()-t0)/60:.1f} min; "
          f"every admission scored exactly once per repeat: {bool(np.isfinite(preds['Stack']).all())}")
    return dict(spec=spec, y=y, g=g, preds=preds, chosen=chosen_log, X=X, prep=prep, splits=pd.DataFrame(split_log))

### **5. Metrics, cluster bootstrap and paired differences**

> **Framework:** E8 Modeling, validation and metrics.

All metrics are computed on the pooled out-of-fold predictions of each repeat and then averaged over repeats.
The bootstrap resamples **patients** (all admissions of a patient move together) and recomputes every metric with
patient-count weights, so intervals reflect the repeated-admission structure. Differences against logistic
regression use the **same** bootstrap samples (paired).

In [16]:
def citl(y, z, w):
    """Calibration-in-the-large: intercept with the slope fixed at 1 (weighted Newton)."""
    a = 0.0
    for _ in range(40):
        pr = expit(z + a)
        g = np.sum(w * (y - pr)); h = np.sum(w * pr * (1 - pr))
        if h < 1e-12:
            break
        step = g / h; a += step
        if abs(step) < 1e-9:
            break
    return a

def calib_slope(y, z, w):
    """Calibration slope (weighted logistic recalibration, Newton)."""
    a, b = 0.0, 1.0
    for _ in range(40):
        pr = expit(a + b * z)
        r = w * (y - pr); v = w * pr * (1 - pr)
        gr = np.array([r.sum(), (r * z).sum()])
        H = np.array([[v.sum(), (v * z).sum()], [(v * z).sum(), (v * z * z).sum()]])
        try:
            dlt = np.linalg.solve(H, gr)
        except np.linalg.LinAlgError:
            break
        a += dlt[0]; b += dlt[1]
        if np.abs(dlt).max() < 1e-9:
            break
    return b

def top_threshold(p, w, k):
    order = np.argsort(-p)
    cw = np.cumsum(w[order])
    idx = int(np.searchsorted(cw, k * cw[-1]))
    return p[order][min(idx, len(p) - 1)]

def metrics_one(y, p, w):
    z = logit(p)
    m = {}
    m["auroc"] = roc_auc_score(y, p, sample_weight=w)
    m["auprc"] = average_precision_score(y, p, sample_weight=w)
    m["brier"] = float(np.average((p - y) ** 2, weights=w))
    m["slope"] = calib_slope(y, z, w)
    m["citl"] = citl(y, z, w)
    m["oe"] = float(np.sum(w * y) / np.sum(w * p))
    edges = np.quantile(p, np.linspace(0, 1, 11)[1:-1])
    bins = np.searchsorted(edges, p)
    W = w.sum(); ece = 0.0
    for b in range(10):
        mk = bins == b; wb = w[mk].sum()
        if wb > 0:
            ece += wb / W * abs(np.sum(w[mk] * y[mk]) / wb - np.sum(w[mk] * p[mk]) / wb)
    m["ece"] = ece
    ev = np.sum(w * y)
    for k in (0.05, 0.10):
        sel = p >= top_threshold(p, w, k)
        m[f"sens_top{int(k*100)}"] = np.sum(w[sel] * y[sel]) / ev
        m[f"ppv_top{int(k*100)}"] = np.sum(w[sel] * y[sel]) / np.sum(w[sel])
    for pt in DCA_T:
        sel = p >= pt
        tp = np.sum(w[sel] * y[sel]); fp = np.sum(w[sel] * (1 - y[sel]))
        m[f"nb_{pt}"] = tp / W - fp / W * pt / (1 - pt)
    return m

def all_metrics(y, preds, w, models, strata):
    """Metrics for each model, averaged over repeats. Returns dict[(model, metric)] and the per-repeat list."""
    per = []
    for r in range(N_REPEATS):
        cur = {}
        for mname in models:
            for k, v in metrics_one(y, preds[mname][r], w).items():
                cur[(mname, k)] = v
        prev = np.sum(w * y) / w.sum()
        for pt in DCA_T:
            cur[("TreatAll", f"nb_{pt}")] = prev - (1 - prev) * pt / (1 - pt)
        for sname, mask in strata.items():
            for mname in ("LR", "Stack"):
                if mname not in models:
                    continue
                wm = w[mask]; ym = y[mask]; pm = preds[mname][r][mask]
                if np.sum(wm * ym) >= 1 and np.sum(wm * (1 - ym)) >= 1:
                    cur[(mname, f"auroc_{sname}")] = roc_auc_score(ym, pm, sample_weight=wm)
                    cur[(mname, f"auprc_{sname}")] = average_precision_score(ym, pm, sample_weight=wm)
                else:
                    cur[(mname, f"auroc_{sname}")] = np.nan
                    cur[(mname, f"auprc_{sname}")] = np.nan
        per.append(cur)
    keys = per[0].keys()
    mean = {k: float(np.nanmean([c[k] for c in per])) if not all(np.isnan(c[k]) for c in per) else np.nan for k in keys}
    return mean, per

def bootstrap(res, B, strata, seed):
    y, g, preds = res["y"], res["g"], res["preds"]
    ug, inv = np.unique(g, return_inverse=True)
    G = len(ug)
    rng = np.random.default_rng(seed)
    out = {}
    t0 = time.time()
    for b in range(B):
        cnt = np.bincount(rng.integers(0, G, G), minlength=G)
        w = cnt[inv].astype(float)
        if np.sum(w * y) < 1:
            continue
        mean, _ = all_metrics(y.astype(float), preds, w, BOOT_MODELS, strata)
        for k, v in mean.items():
            out.setdefault(k, []).append(v)
        for a_, b_ in PAIRS:            # paired differences (same bootstrap sample)
            for met in ("auroc", "auprc", "brier") + tuple(f"nb_{pt}" for pt in DCA_T):
                out.setdefault((f"{a_}-{b_}", met), []).append(mean[(a_, met)] - mean[(b_, met)])
        if b == 4:
            el = time.time() - t0
            print(f"  bootstrap: 5 replicates took {el:.0f} s -> projected {el*B/5/60:.1f} min for B={B}")
    return {k: np.array(v) for k, v in out.items()}

### **6. Reporting function**

> **Framework:** E8 and E11 Reporting. Aggregate reports only.

Prints aggregate tables only: discrimination, calibration, paired differences against logistic regression, the share of
events captured in the top 5 % / 10 % of predicted risk, net benefit, development instability across repeats and the
hyperparameters chosen in the inner loops.

In [17]:
def _ci(boot, key, d=3):
    a = boot.get(key)
    if a is None or len(a) < 20:
        return ""
    lo, hi = np.nanpercentile(a, [2.5, 97.5])
    return f" ({lo:.{d}f}-{hi:.{d}f})"

def decile_table(res, models=("LR", "Stack")):
    y = res["y"].astype(float)
    out = None
    for m in models:
        acc = []
        for r in range(N_REPEATS):
            p = res["preds"][m][r]
            edges = np.quantile(p, np.linspace(0, 1, 11)[1:-1])
            b = np.searchsorted(edges, p)
            acc.append(pd.DataFrame({"b": b, "p": p, "y": y}).groupby("b").agg(n=("y", "size"), pred=("p", "mean"), obs=("y", "mean")))
        mean = sum(acc) / len(acc)
        t = pd.DataFrame({"decile": mean.index + 1, "n": mean["n"].round(0).astype(int)}) if out is None else out
        t[f"{m}_pred"] = mean["pred"].round(4).values
        t[f"{m}_obs"] = mean["obs"].round(4).values
        out = t
    return out

def plot_calibration(res):
    import matplotlib.pyplot as plt
    t = decile_table(res)
    fig, ax = plt.subplots(figsize=(5, 5))
    mx = float(max(t["LR_pred"].max(), t["LR_obs"].max(), t["Stack_pred"].max(), t["Stack_obs"].max())) * 1.1
    ax.plot([0, mx], [0, mx], color="grey", lw=1, ls="--", label="ideal")
    ax.plot(t["LR_pred"], t["LR_obs"], "o-", label="Logistic regression")
    ax.plot(t["Stack_pred"], t["Stack_obs"], "s-", label="Stacked ensemble")
    ax.set_xlabel("Mean predicted risk (decile)"); ax.set_ylabel("Observed SMM rate")
    ax.set_title(f"Calibration by decile -- {res['spec']['name']}"); ax.legend(); plt.show()   # displayed only, not saved

def emit(name, fn):
    """Print the aggregate report and also keep it as a text file (aggregates only) so it can be shared without truncation."""
    import io, contextlib
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        fn()
    text = buf.getvalue()
    print(text)
    path = f"../data/outputs_{name}.txt"
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(text)
    print(f"[aggregate report saved as data/{os.path.basename(path)}  ({len(text)} characters)]")

def run_analysis(name, B):
    spec = make_spec(name)
    res = run_cv(spec)
    d = spec["data"]
    strata = {}
    if name == "main":
        strata = {"icu1": d["icu_in_24h"].values.astype(bool), "icu0": ~d["icu_in_24h"].values.astype(bool)}
    y = res["y"].astype(float)
    res["point"], res["per"] = all_metrics(y, res["preds"], np.ones(len(y)), MODELS + [], strata)
    res["boot"] = bootstrap(res, B, strata, SEED + 7)
    res["strata"] = strata
    return res

def report(res):
    sp = res["spec"]; d = sp["data"]; y = res["y"]
    pt, per, bt = res["point"], res["per"], res["boot"]
    n, ev, npat = len(y), int(y.sum()), d.subject_id.nunique()
    print("=" * 110)
    print(f"ANALYSIS {sp['name']}: {n} admissions, {npat} patients, {ev} events ({100*ev/n:.2f}%), "
          f"numeric={len(sp['num'])}, binary={len(sp['bin'])}, categorical={len(sp['cat'])}")
    print(f"Repeats={N_REPEATS}, outer folds={OUTER_FOLDS}, inner folds={INNER_FOLDS}, bootstrap B={len(next(iter(bt.values()))) if bt else 0}, QUICK={QUICK}")
    print("=" * 110)

    sl = res["splits"]
    print("\n[0] Patients per outer partition (repeat 1 shown; every split verified to share no patient between training and test)")
    print(sl[sl.repeat == 1].drop(columns=["repeat"]).to_string(index=False))
    print(f"    Across all {len(sl)} splits: test patients per fold {sl.test_pat.min()}-{sl.test_pat.max()}, "
          f"event-bearing test patients per fold {sl.test_event_pat.min()}-{sl.test_event_pat.max()}, "
          f"patients present on both sides: {int(sl.shared_patients.sum())}")

    rows = []
    for m in MODELS:
        r = {"model": m}
        for met, dd in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
            r[met] = f"{pt[(m, met)]:.{dd}f}" + (_ci(bt, (m, met), dd) if m in BOOT_MODELS else "")
        r["sd_auroc_repeats"] = f"{np.std([c[(m,'auroc')] for c in per], ddof=1):.4f}" if len(per) > 1 else "n/a"
        r["sd_auprc_repeats"] = f"{np.std([c[(m,'auprc')] for c in per], ddof=1):.4f}" if len(per) > 1 else "n/a"
        rows.append(r)
    print("\n[1] Discrimination and overall accuracy (mean over repeats; 95% cluster-bootstrap CI)")
    print(f"    prevalence = {y.mean():.4f}  (AUPRC of a non-informative model equals the prevalence)")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for m in MODELS:
        r = {"model": m}
        for met, dd in [("slope", 2), ("citl", 2), ("oe", 2), ("ece", 4)]:
            r[met] = f"{pt[(m, met)]:.{dd}f}" + (_ci(bt, (m, met), dd) if m in BOOT_MODELS else "")
        rows.append(r)
    print("\n[2] Calibration (slope ideal 1; calibration-in-the-large ideal 0; O/E ideal 1; ECE with 10 quantile bins)")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for a_, b_ in PAIRS:
        r = {"comparison (first minus second)": f"{a_} - {b_}"}
        for met, dd in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
            r[met] = f"{pt[(a_, met)] - pt[(b_, met)]:+.{dd}f}" + _ci(bt, (f"{a_}-{b_}", met), dd)
        rows.append(r)
    print("\n[3] Paired differences (same bootstrap samples); the last row compares the ensemble with XGBoost")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for m in BOOT_MODELS:
        rows.append({"model": m,
                     "sens_top5%": f"{pt[(m,'sens_top5')]:.3f}" + _ci(bt, (m, "sens_top5")),
                     "ppv_top5%": f"{pt[(m,'ppv_top5')]:.3f}",
                     "sens_top10%": f"{pt[(m,'sens_top10')]:.3f}" + _ci(bt, (m, "sens_top10")),
                     "ppv_top10%": f"{pt[(m,'ppv_top10')]:.3f}"})
    print("\n[4] Events captured among the 5% / 10% of admissions with the highest predicted risk")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for m in BOOT_MODELS + ["TreatAll"]:
        rows.append({"strategy": m, **{f"pt={t:g}": round(pt[(m, f'nb_{t}')], 5) for t in DCA_T}})
    rows.append({"strategy": "TreatNone", **{f"pt={t:g}": 0.0 for t in DCA_T}})
    print("\n[5] Net benefit (decision curve) at the stated threshold probabilities")
    print(pd.DataFrame(rows).to_string(index=False))
    rows = []
    for a_, b_ in [("Stack", "LR"), ("RF", "LR"), ("ET", "LR"), ("XGB", "LR"), ("Stack", "XGB")]:
        rows.append({"comparison": f"{a_} - {b_}", **{f"pt={t:g}": f"{pt[(a_,f'nb_{t}')]-pt[(b_,f'nb_{t}')]:+.5f}" + _ci(bt, (f"{a_}-{b_}", f"nb_{t}"), 5) for t in DCA_T[:6]}})
    print("    Paired net-benefit differences:")
    print(pd.DataFrame(rows).to_string(index=False))

    if res["strata"]:
        rows = []
        for sname, lab in [("icu1", "ICU admission within 24 h"), ("icu0", "no ICU within 24 h")]:
            mk = res["strata"][sname]
            r = {"stratum": lab, "n": int(mk.sum()), "events": int(y[mk].sum())}
            for m in ("LR", "Stack"):
                r[f"{m}_auroc"] = (f"{pt[(m, 'auroc_' + sname)]:.3f}" + _ci(bt, (m, "auroc_" + sname))) if not np.isnan(pt[(m, 'auroc_' + sname)]) else "n/a"
                r[f"{m}_auprc"] = (f"{pt[(m, 'auprc_' + sname)]:.3f}") if not np.isnan(pt[(m, 'auprc_' + sname)]) else "n/a"
            rows.append(r)
        print("\n[6] Performance by ICU admission within the first 24 h (same predictions, stratified evaluation)")
        print(pd.DataFrame(rows).to_string(index=False))

    print("\n[8] Calibration by decile of predicted risk (outer-test predictions; mean over repeats)")
    print(decile_table(res).to_string(index=False))

    ch = pd.DataFrame(res["chosen"], columns=["learner", "config"])
    tab = ch.value_counts().rename("outer_fits").reset_index().sort_values(["learner", "outer_fits"], ascending=[True, False])
    print(f"\n[7] Hyperparameters selected in the inner loops (out of {N_REPEATS*OUTER_FOLDS} outer fits per learner)")
    print(tab.to_string(index=False))

### **7. Develop on the earlier period, test once on the later period**

> **Framework:** E9 · gate-P4 criterion C4: discriminates (temporal AUROC of 0.70 or more) and calibration correctable by the intercept alone. `cut2017_landmark_labs` serves return R4 and `cut2017_escalation` return R2.

`run_temporal` develops the full pipeline on the development set (`tune_stack_calibrate`, the same function as notebooks 02 and 08) and
scores the test set. `report_temporal` prints discrimination with patient-level bootstrap intervals, calibration (slope, calibration-in-the-large,
O/E, ECE), paired differences between models, events captured in the top 5 % / 10 %, net benefit and calibration by decile.

In [18]:
def run_temporal(spec):
    tr, te = spec["train"], spec["test"]
    Xtr = prep_frame(tr, spec["num"], spec["bin"], spec["cat"])
    Xte = prep_frame(te, spec["num"], spec["bin"], spec["cat"])
    ytr = tr[spec["label"]].values.astype(int); gtr = tr["subject_id"].values
    yte = te[spec["label"]].values.astype(int); gte = te["subject_id"].values
    assert not set(gtr) & set(gte), "patient present in development and test"
    prep = make_prep(spec["num"], spec["bin"], spec["cat"])
    preds = {m: np.full((N_REPEATS, len(yte)), np.nan) for m in MODELS}
    chosen = []
    t0 = time.time()
    for r in range(N_REPEATS):
        out, ch = tune_stack_calibrate(Xtr, ytr, gtr, Xte, prep, SEED + 1000 * r)
        for m in MODELS:
            preds[m][r] = out[m]
        chosen += [(n, cfg_str(c)) for n, c in ch.items()]
        print(f"[{spec['name']}] repeat {r + 1}/{N_REPEATS} done ({(time.time() - t0) / 60:.1f} min)")
    res = dict(spec=spec, y=yte, g=gte, preds=preds, chosen=chosen, ytr=ytr)
    res["point"], res["per"] = all_metrics(yte.astype(float), preds, np.ones(len(yte)), MODELS, {})
    res["boot"] = bootstrap(res, BOOT_SENS, {}, SEED + 7)
    return res

def report_temporal(res):
    sp, i = res["spec"], res["spec"]["info"]
    pt, per, bt, y = res["point"], res["per"], res["boot"], res["y"]
    print("=" * 110)
    print(f"TEMPORAL VALIDATION {sp['name']} (cut {i['cut']}), QUICK={QUICK}, repeats={N_REPEATS}, bootstrap B={BOOT_SENS}")
    print("=" * 110)
    print(f"Development (year_hi < {i['cut']}): {i['dev_adm']} admissions, {i['dev_pat']} patients, {i['dev_events']} events "
          f"(prevalence {100 * i['dev_events'] / i['dev_adm']:.2f}%)")
    print(f"Discarded (interval straddles {i['cut']}): {i['straddling']} admissions, {i['straddling_events']} events")
    print(f"Test (year_lo >= {i['cut']}): {i['test_adm_all']} admissions; removed because the patient is in development: "
          f"{i['test_removed_shared']} ({i['test_removed_shared_events']} events)")
    print(f"Test analysed: {i['test_adm']} admissions, {i['test_pat']} patients, {i['test_events']} events "
          f"(prevalence {100 * i['test_events'] / i['test_adm']:.2f}%)")

    rows = []
    for m in MODELS:
        r = {"model": m}
        for met, dd in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
            r[met] = f"{pt[(m, met)]:.{dd}f}" + (_ci(bt, (m, met), dd) if m in BOOT_MODELS else "")
        r["sd_auroc_repeats"] = f"{np.std([c[(m, 'auroc')] for c in per], ddof=1):.4f}" if len(per) > 1 else "n/a"
        rows.append(r)
    print("\n[1] Discrimination in the later period (mean over repeats; 95% patient-level bootstrap CI)")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for m in MODELS:
        r = {"model": m}
        for met, dd in [("slope", 2), ("citl", 2), ("oe", 2), ("ece", 4)]:
            r[met] = f"{pt[(m, met)]:.{dd}f}" + (_ci(bt, (m, met), dd) if m in BOOT_MODELS else "")
        r["mean predicted"] = f"{np.mean(res['preds'][m]):.4f}"
        rows.append(r)
    print(f"\n[2] Calibration in the later period (observed prevalence {y.mean():.4f}; O/E > 1 = risk underestimated)")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for a_, b_ in PAIRS:
        r = {"comparison": f"{a_} - {b_}"}
        for met, dd in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
            r[met] = f"{pt[(a_, met)] - pt[(b_, met)]:+.{dd}f}" + _ci(bt, (f"{a_}-{b_}", met), dd)
        rows.append(r)
    print("\n[3] Paired differences (same bootstrap samples)")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = []
    for m in BOOT_MODELS:
        rows.append({"model": m, "sens_top5%": f"{pt[(m, 'sens_top5')]:.3f}" + _ci(bt, (m, "sens_top5")),
                     "ppv_top5%": f"{pt[(m, 'ppv_top5')]:.3f}",
                     "sens_top10%": f"{pt[(m, 'sens_top10')]:.3f}" + _ci(bt, (m, "sens_top10")),
                     "ppv_top10%": f"{pt[(m, 'ppv_top10')]:.3f}"})
    print("\n[4] Events captured among the 5% / 10% of admissions with the highest predicted risk")
    print(pd.DataFrame(rows).to_string(index=False))

    rows = [{"strategy": m, **{f"pt={t:g}": round(pt[(m, f'nb_{t}')], 5) for t in DCA_T}} for m in BOOT_MODELS + ["TreatAll"]]
    print("\n[5] Net benefit by threshold probability")
    print(pd.DataFrame(rows).to_string(index=False))

    print("\n[6] Calibration by decile of predicted risk (mean over repeats)")
    print(decile_table(res).to_string(index=False))

    ch = pd.DataFrame(res["chosen"], columns=["model", "config"]).value_counts().rename("times chosen").reset_index()
    print("\n[7] Hyperparameters chosen in the inner loops of the development set")
    print(ch.sort_values(["model", "times chosen"], ascending=[True, False]).to_string(index=False))

In [19]:
TEMPORAL_RUN = list(TEMPORAL)   # remove names to run only some analyses
RESULTS = {}
for nm in TEMPORAL_RUN:
    sp = make_temporal(nm)
    for side in ("train", "test"):
        ev = int(sp[side][sp["label"]].sum())
        if ev < 40:
            print(f"WARNING: {nm} has only {ev} events in the {side} set; report as exploratory.")
    RESULTS[nm] = run_temporal(sp)
    emit(nm, lambda r=RESULTS[nm]: report_temporal(r))

[cut2017] repeat 1/3 done (0.5 min)
[cut2017] repeat 2/3 done (1.0 min)
[cut2017] repeat 3/3 done (1.5 min)
  bootstrap: 5 replicates took 0 s -> projected 0.4 min for B=300
TEMPORAL VALIDATION cut2017 (cut 2017), QUICK=False, repeats=3, bootstrap B=300
Development (year_hi < 2017): 11104 admissions, 8822 patients, 209 events (prevalence 1.88%)
Discarded (interval straddles 2017): 1964 admissions, 35 events
Test (year_lo >= 2017): 6315 admissions; removed because the patient is in development: 1240 (28 events)
Test analysed: 5075 admissions, 4202 patients, 173 events (prevalence 3.41%)

[1] Discrimination in the later period (mean over repeats; 95% patient-level bootstrap CI)
    model               auroc               auprc                     brier sd_auroc_repeats
       LR 0.744 (0.705-0.778) 0.154 (0.114-0.208) 0.03114 (0.02702-0.03549)           0.0009
       RF 0.745 (0.707-0.784) 0.177 (0.127-0.241) 0.03083 (0.02665-0.03469)           0.0018
       ET 0.738 (0.702-0.774) 0.122 

### **8. Summary table for the manuscript (aggregates only)**

> **Framework:** E11 Reporting and gate P4 (D20). One row per analysis and model for the manuscript and the classification.

One row per analysis and model (logistic regression, XGBoost, stacked ensemble): development and test sizes, prevalence in each period,
AUROC and AUPRC with 95 % intervals, calibration slope and O/E in the later period. Saved as `data/outputs_temporal_summary.txt`.

In [20]:
def temporal_summary():
    rows = []
    for nm, res in RESULTS.items():
        i, pt, bt = res["spec"]["info"], res["point"], res["boot"]
        for m in ("LR", "XGB", "Stack"):
            rows.append({"analysis": nm, "model": m,
                         "dev (events, prev %)": f"{i['dev_adm']} ({i['dev_events']}, {100 * i['dev_events'] / i['dev_adm']:.2f})",
                         "test (events, prev %)": f"{i['test_adm']} ({i['test_events']}, {100 * i['test_events'] / i['test_adm']:.2f})",
                         "auroc": f"{pt[(m, 'auroc')]:.3f}" + _ci(bt, (m, "auroc")),
                         "auprc": f"{pt[(m, 'auprc')]:.3f}" + _ci(bt, (m, "auprc")),
                         "slope": f"{pt[(m, 'slope')]:.2f}" + _ci(bt, (m, "slope"), 2),
                         "O/E": f"{pt[(m, 'oe')]:.2f}" + _ci(bt, (m, "oe"), 2)})
    print(f"TEMPORAL VALIDATION SUMMARY (QUICK={QUICK})")
    print(pd.DataFrame(rows).to_string(index=False))

emit("temporal_summary", temporal_summary)

TEMPORAL VALIDATION SUMMARY (QUICK=False)
             analysis model dev (events, prev %) test (events, prev %)               auroc               auprc            slope              O/E
              cut2017    LR    11104 (209, 1.88)      5075 (173, 3.41) 0.744 (0.705-0.778) 0.154 (0.114-0.208) 0.79 (0.68-0.92) 1.27 (1.09-1.46)
              cut2017   XGB    11104 (209, 1.88)      5075 (173, 3.41) 0.747 (0.707-0.783) 0.146 (0.111-0.205) 0.83 (0.71-0.97) 1.34 (1.15-1.52)
              cut2017 Stack    11104 (209, 1.88)      5075 (173, 3.41) 0.752 (0.713-0.789) 0.169 (0.126-0.229) 0.84 (0.72-0.98) 1.31 (1.12-1.49)
              cut2015    LR     8017 (121, 1.51)      7175 (214, 2.98) 0.764 (0.734-0.794) 0.135 (0.104-0.185) 0.90 (0.80-1.03) 1.41 (1.23-1.60)
              cut2015   XGB     8017 (121, 1.51)      7175 (214, 2.98) 0.760 (0.724-0.792) 0.120 (0.096-0.164) 0.92 (0.82-1.05) 1.48 (1.30-1.68)
              cut2015 Stack     8017 (121, 1.51)      7175 (214, 2.98) 0.766 (0.731-0.79